In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]

In [0]:
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("HealthcarePipeline").getOrCreate()
df = spark.createDataFrame(data, columns)
df.show()


In [0]:
df2 = df.withColumn(
    "total_bill",
    col("consultation_fee") + (col("tests_count") * 500)
).withColumn(
    "patient_type",
    when(col("total_bill") >= 6000, "High")
    .when(col("total_bill") >= 4000, "Medium")
    .otherwise("Low")
)
df2.show()

In [0]:
high_value_df = df2.filter(col("patient_type") == "High")
high_value_df.show()

In [0]:
agg_df = df2.groupBy("department").agg(
    count("*").alias("total_patients"),
    sum("total_bill").alias("total_revenue"),
    avg("total_bill").alias("avg_bill")
)
agg_df.show()

In [0]:
sorted_df = agg_df.orderBy(col("total_revenue").desc())
sorted_df.show()

Part 2

In [0]:
df2.createOrReplaceTempView("patient_visits")

In [0]:
%sql
SELECT * FROM patient_visits WHERE department = 'Cardiology';


In [0]:
%sql
SELECT city,SUM(total_bill) AS total_revenue
FROM patient_visits
GROUP BY city
ORDER BY total_revenue DESC;

In [0]:
%sql
SELECT * FROM patient_visits
ORDER BY total_bill DESC
LIMIT 5;

In [0]:
%sql
SELECT department,COUNT(*) AS patient_count
FROM patient_visits
GROUP BY department
ORDER BY patient_count DESC;

Delta Lake


In [0]:
%sql
-- 1. Create a Delta Table from dataset
CREATE TABLE patient_delta
USING DELTA
AS
SELECT * FROM patient_visits;

In [0]:
%sql
INSERT INTO patient_delta VALUES
(109,"sureshkrishna","Chennai","Cardiology",2000,2,6000,"High"),
(110,"ramesh","Chennai","Neurology",3000,1,5500,"High");

In [0]:
%sql
UPDATE patient_delta
SET consultation_fee = consultation_fee + 500
WHERE department = 'Cardiology';

In [0]:
%sql
DELETE FROM patient_delta
WHERE patient_type = 'Low';

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW updates AS
SELECT * FROM VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",5500,2,6500,"High"),
(111,"Neha Sharma","Delhi","Dermatology",2000,1,2500,"Low")
AS updates(
visit_id,
patient_name,
city,
department,
consultation_fee,
tests_count,
total_bill,
patient_type
);

MERGE INTO patient_delta AS target
USING updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
UPDATE SET
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.total_bill = source.total_bill,
target.patient_type = source.patient_type

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql
DESCRIBE HISTORY patient_delta;

In [0]:
%sql
SELECT *
FROM patient_delta VERSION AS OF 0;

In [0]:
%sql VACUUM patient_delta RETAIN 168 HOURS DRY RUN;

In [0]:
df2.write.mode("overwrite").saveAsTable("patient_parquet")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE patient_delta
USING DELTA
AS SELECT * FROM patient_parquet
""")

In [0]:
spark.sql("SELECT * FROM patient_delta").show()
spark.sql("DESCRIBE DETAIL patient_delta").show()

In [0]:
%sql
CREATE TABLE patient_target
USING DELTA
AS
SELECT * FROM patient_visits;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",6000,2,7000,"High"), 
(112,"Pooja Shah","Mumbai","Neurology",8000,1,8500,"High") 
AS daily_updates(
visit_id,
patient_name,
city,
department,
consultation_fee,
tests_count,
total_bill,
patient_type
);

In [0]:
%sql
MERGE INTO patient_target AS target
USING daily_updates AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
UPDATE SET
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.total_bill = source.total_bill,
target.patient_type = source.patient_type

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql
create catalog new_healthcare_catalog;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7009197683871897>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'create catalog new_healthcare_catalog;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.driver_activity_logger

In [0]:
%sql
create schema new_healthcare_catalog.hospital_schema;

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-7579314531966858>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'create schema new_healthcare_catalog.hospital_schema;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.driver_

In [0]:
%sql
use catalog new_healthcare_catalog;
use schema hospital_schema;

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-7579314531966859>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'use catalog new_healthcare_catalog;\nuse schema hospital_schema;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     s

In [0]:
%sql
create table patient_records (
visit_id int,
patient_name string,
city string,
department string,
total_bill double
);

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-7579314531966860>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'DROP CATALOG new_healthcare_catalog;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.driver_activity_logger.l

In [0]:
%sql
insert into patient_records values
(1005,'ram','TamilNadu','Sidhha',5000),
(1000,'madhesh','Delhi','Homeopathy',4000)


In [0]:
%sql
select * from patient_records;

In [0]:
new_data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

In [0]:
new_df = spark.createDataFrame(new_data, columns)
new_df.createOrReplaceTempView("new_data")

In [0]:
%sql
merge into patient_delta as target
using new_data as source
on target.visit_id = source.visit_id

when matched then
update set
  target.patient_name = source.patient_name,
  target.city = source.city,
  target.department = source.department,
  target.consultation_fee = source.consultation_fee,
  target.tests_count = source.tests_count

when not matched then
insert (
  visit_id,
  patient_name,
  city,
  department,
  consultation_fee,
  tests_count
)
values (
  source.visit_id,
  source.patient_name,
  source.city,
  source.department,
  source.consultation_fee,
  source.tests_count
);

In [0]:
%sql
use catalog hexa_ws_7405609128804000;
use schema default;

create table new_patient_records as
select * from patient_delta;

In [0]:
spark.sql("""
grant select on table patient_records 
to `azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com`
""")